In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader


# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # SubtractOne()
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=True)

# Write your code here
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:4]))


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Write your code here
efficientnet = efficientnet_v2_s(weights=torchvision.models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
efficientnet.requires_grad_(False)
efficientnet.classifier.requires_grad_(True)
efficientnet.classifier[1] = nn.Linear(efficientnet.classifier[1].in_features, num_classes)


In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(efficientnet.classifier.parameters(), lr=0.001)
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.to(device)
    total_acc = 0
    total_loss = 0
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        # -1 for IndexError: Target 26 is out of bounds.
        labels.cpu().numpy()
        labels -= 1
        labels.to(device)
        optimizer.zero_grad() # Zero the parameter gradients

        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(dataloader), total_acc / len(dataloader)

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    model.to(device)
    total_acc = 0
    total_loss = 0
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            # -1 for IndexError: Target 26 is out of bounds.
            labels.cpu().numpy()
            labels -= 1
            labels.to(device)

            logits = model(inputs)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(dataloader), total_acc / len(dataloader)


In [ ]:

criterion = nn.CrossEntropyLoss()
learning_rate = 0.01


optimizer = torch.optim.AdamW(efficientnet.parameters(), learning_rate)

num_epochs = 1

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(efficientnet, train_loader, criterion, optimizer, device)
    test_loss, test_acc = validate_epoch(efficientnet, test_loader, criterion, device)


    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
